# Часть 1 — A2A: как агенты разговаривают друг с другом

В занятии 2 мы научили агента пользоваться **инструментами** через MCP. Теперь агентов станет
несколько, и им нужно общаться между собой. Для этого есть открытый протокол **A2A (Agent2Agent)**.

| | MCP | A2A |
|---|---|---|
| кто с кем | агент ↔ **инструмент** | агент ↔ **агент** |
| что на той стороне | функция: дал аргументы — получил результат | другой агент: своя модель, свои инструменты, свои рассуждения |
| что видно снаружи | список инструментов и их аргументы | только «визитка» агента и его ответ |
| как узнать возможности | `tools/list` | **карточка агента** `/.well-known/agent-card.json` |
| единица работы | вызов инструмента | **задача (Task)** со статусами — может идти долго |

Главные понятия A2A:

- **Agent Card** — визитка: имя, описание, что умеет (skills), адрес;
- **Message** — сообщение: роль (пользователь / агент) и части (текст, данные, файлы);
- **Task** — задача, которую агент выполняет по сообщению: `submitted → working → completed`;
- **Artifact** — результат задачи (у нас — текст ответа).

В этом блокноте наш **аналитик банка** из занятия 2 становится A2A-агентом, и мы поговорим с ним:
сначала «сырыми» HTTP-запросами, потом через SDK.

**Перед запуском** должны работать (каждое — в своём терминале, из папки занятия):

```bash
docker compose up -d                  # база банка + Langfuse (один раз)
uv run python mcp_server.py           # терминал 1: MCP-сервер с инструментами → :8101
uv run python analyst_agent.py        # терминал 2: агент-аналитик по A2A     → :8102
```

In [1]:
import httpx

for name, url in [("MCP-сервер", "http://localhost:8101/mcp"), ("аналитик", "http://localhost:8102/.well-known/agent-card.json")]:
    try:
        httpx.get(url, timeout=3)
        print(f"✅ {name} работает")
    except httpx.ConnectError:
        print(f"❌ {name} не запущен — см. команды выше")

✅ MCP-сервер работает
✅ аналитик работает


## Шаг 1. Карточка агента

Любой A2A-агент публикует свою визитку по стандартному адресу. По ней другой агент (или человек)
понимает, к кому обращаться и с чем.

In [2]:
import json

card = httpx.get("http://localhost:8102/.well-known/agent-card.json").json()
print(json.dumps(card, indent=2, ensure_ascii=False))

{
  "name": "Bank analyst",
  "description": "Отвечает на вопросы по данным банка: счета, обороты, балансы, кредиты, клиенты.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8102/",
      "protocolBinding": "JSONRPC"
    }
  ],
  "version": "1.0",
  "capabilities": {
    "streaming": false
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "bank-data",
      "name": "Данные банка",
      "description": "Факты и цифры по базе банка: SQL-аналитика по счетам, транзакциям, кредитам.",
      "tags": [
        "bank",
        "sql",
        "analytics"
      ],
      "examples": [
        "Какой средний месячный приход на счёт 2378 за 1998 год?",
        "Есть ли у счёта 2378 кредиты и в каком они статусе?"
      ]
    }
  ],
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3",
  "url": "http://localhost:8102/"
}


- `name`, `description` — кто это;
- `skills` — что умеет, с примерами вопросов: по ним агент-заказчик решает, кому что поручить;
- `supportedInterfaces` — куда и по какому протоколу слать сообщения (у нас JSON-RPC на `/`);
- `capabilities` — поддерживает ли стриминг, уведомления и т. п.;
- `preferredTransport`, `protocolVersion: 0.3`, `url` — те же сведения в формате старой версии протокола
  (SDK добавляет их сам, чтобы со стороны старых клиентов агент тоже был виден).

Обратите внимание, чего в карточке **нет**: модели, инструментов, SQL, базы. Аналитик — «чёрный ящик».

## Шаг 2. Сообщение «руками»

A2A — это JSON-RPC поверх HTTP (как MCP по HTTP). Метод `SendMessage`, в параметрах — сообщение.
Отправим вопрос обычным `httpx` и посмотрим на ответ целиком. Аналитику нужно несколько секунд:
он сам смотрит таблицы и пишет SQL — следите за терминалами 1 и 2.

In [4]:
import uuid

request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "SendMessage",
    "params": {
        "message": {
            "role": "ROLE_USER",
            "messageId": str(uuid.uuid4()),
            "parts": [{"text": "Сколько всего клиентов в банке?"}],
        }
    },
}

response = httpx.post("http://localhost:8102/", json=request, headers={"A2A-Version": "1.0"}, timeout=120).json()
print(json.dumps(response, indent=2, ensure_ascii=False))

{
  "result": {
    "task": {
      "id": "af08a86e-b9ae-469f-bdd8-b4433b5a922e",
      "contextId": "a69d87ac-7b63-400d-95c0-b50d711dab0e",
      "status": {
        "state": "TASK_STATE_COMPLETED",
        "timestamp": "2026-10-15T17:10:03.960135Z"
      },
      "artifacts": [
        {
          "artifactId": "51965108-b2af-4c18-81fd-481191f5d64e",
          "name": "answer",
          "parts": [
            {
              "text": "В банке **5 369 клиентов**."
            }
          ]
        }
      ],
      "history": [
        {
          "messageId": "15b41850-6440-42db-b44c-7e1c51b276a7",
          "contextId": "a69d87ac-7b63-400d-95c0-b50d711dab0e",
          "taskId": "af08a86e-b9ae-469f-bdd8-b4433b5a922e",
          "role": "ROLE_USER",
          "parts": [
            {
              "text": "Сколько всего клиентов в банке?"
            }
          ]
        }
      ]
    }
  },
  "id": 1,
  "jsonrpc": "2.0"
}


Что вернулось — **задача** (`result.task`):

- `id` — номер задачи, `contextId` — номер разговора (несколько задач могут быть в одном контексте);
- `status.state` — `TASK_STATE_COMPLETED`: задача выполнена. Для долгих задач клиент может получить
  `TASK_STATE_WORKING` и потом спросить статус (`GetTask`) или подписаться на обновления;
- `artifacts` — результат: текст ответа аналитика;
- `history` — сообщения задачи (наш вопрос).

Достанем сам ответ:

In [5]:
task = response["result"]["task"]
print("статус:", task["status"]["state"])
print("ответ: ", task["artifacts"][0]["parts"][0]["text"])

статус: TASK_STATE_COMPLETED
ответ:  В банке **5 369 клиентов**.


## Шаг 3. То же через SDK

Руками JSON писать не нужно: в `a2a-sdk` есть клиент. Он сам читает карточку агента, выбирает
протокол и собирает сообщение.

In [6]:
from a2a.client import ClientConfig, create_client
from a2a.helpers.proto_helpers import get_artifact_text, new_text_part
from a2a.types import Message, Role, SendMessageRequest, TaskState

client = await create_client(
    "http://localhost:8102",                                   # by the address it finds the agent card
    client_config=ClientConfig(streaming=False, httpx_client=httpx.AsyncClient(timeout=300)),
)

message = Message(role=Role.ROLE_USER, message_id=str(uuid.uuid4()),
                  parts=[new_text_part("Есть ли у счёта 2378 кредиты?")])

async for event in client.send_message(SendMessageRequest(message=message)):
    print("статус:", TaskState.Name(event.task.status.state))
    print("ответ: ", get_artifact_text(event.task.artifacts[0]))

await client.close()

статус: TASK_STATE_COMPLETED
ответ:  У счёта 2378 кредитов нет.


Если что-то пошло не так (например, у аналитика упала модель), задача придёт со статусом
`TASK_STATE_FAILED`, а причина — в `task.status.message`.

## Шаг 4. Как аналитик стал A2A-агентом

Откройте `analyst_agent.py`. В нём три части:

1. **карточка** `CARD` — то, что мы получили в шаге 1;
2. **сам агент** `build_graph` — тот же граф «агент ⇄ инструменты» из занятия 2, инструменты — с
   MCP-сервера `mcp_server.py`;
3. **обёртка A2A** `AnalystExecutor` — получила сообщение → запустила граф → вернула ответ.

Вот обёртка целиком:

In [ ]:
import inspect

import analyst_agent

print(inspect.getsource(analyst_agent.AnalystExecutor.execute))

- `context.get_user_input()` — текст входящего сообщения;
- `TaskUpdater` двигает задачу по статусам: `start_work()` → `add_artifact(...)` → `complete()`
  (а при ошибке — `failed(...)` с причиной);
- `self.answer(...)` запускает граф (внутри — ещё и Langfuse, до него дойдём в части 3).

Остальное (HTTP, JSON-RPC, хранение задач) делает SDK: `DefaultRequestHandler` + маршруты в конце файла.

**Главная мысль:** граф агента не меняется. Любого агента можно сделать A2A-сервисом, добавив
карточку и маленькую обёртку — и с ним сможет говорить агент на любом фреймворке и языке.

## Итоги

- A2A — протокол «агент ↔ агент»: карточка (что умею) + сообщения + задачи со статусами;
- снаружи агент — чёрный ящик: модель, инструменты и данные остаются у его владельца;
- под капотом — JSON-RPC по HTTP; в Python удобнее клиент `a2a-sdk`.

Дальше — часть 2: второй агент, который сам обращается к аналитику.